# HQDR-Net: Hybrid Quantum Data Re-uploading Network
## UAV Wildfire Detection — Full Kaggle Training Pipeline
**Reg No:** 22BCB0202 | **Guide:** Prof. Sathya K | **VIT University**

## Cell 1: Install Dependencies
**Lines:** pip install torch torchvision torchcam pennylane

In [ ]:
!pip install torch torchvision torchcam qiskit==0.45.2 qiskit-machine-learning==0.7.1 pennylane onnxruntime -q

## Cell 2: Imports & Environment
**Lines 15–54** — All library imports, random seeds, device selection

In [ ]:
import os, glob, time, random, shutil, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

import pennylane as qml

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, confusion_matrix,
    roc_curve, precision_recall_curve, auc)
from torchcam.methods import GradCAM

import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Cell 3: Configuration
**Lines 61–90** — All hyperparameters: backbone, qubits, layers, LR, epochs

In [ ]:
CFG = {
    'dataset_root': '/kaggle/input/datasets/dharunr162/uav-firedataset',
    'img_size': 224,
    'num_classes': 2,
    'train_ratio': 0.70,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    'backbone': 'mobilenet_v3_small',
    'backbone_feature_dim': 576,
    'n_qubits': 8,
    'n_reuploading_layers': 3,
    'quantum_output_dim': 8,
    'batch_size': 16,
    'epochs': 15,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'label_smoothing': 0.05,
    'freeze_until_layer': 0,
    'checkpoint_path': 'best_model.pt',
}
print("Configuration loaded")

## Cell 4: Dataset Exploration
**Lines 97–129** — Folder path mapping, image collection, class counts

In [ ]:
BASE_FOLDER = "UAVS-FDDB UAVs-based Forest Fire Detection Database"

FOLDER_LABEL_MAP = {
    f"{BASE_FOLDER}/Original Image Dataset (Raw Images)/Evening Fire Incident_raw_img/Evening Fire Incident_raw_img": 1,
    f"{BASE_FOLDER}/Original Image Dataset (Raw Images)/Pre-Evening Fire Incident_raw_img/Pre-Evening Fire Incident_raw_img": 1,
    f"{BASE_FOLDER}/Original Image Dataset (Raw Images)/Evening Forest condition_raw_img/Evening Forest condition_raw_img": 0,
    f"{BASE_FOLDER}/Original Image Dataset (Raw Images)/Pre-evening Forest condition_raw_img/Pre-evening Forest condition_raw_img": 0,
    f"{BASE_FOLDER}/Augmented Images/Evening_Fire_Incident_aug_img/Evening_Fire_Incident_aug_img": 1,
    f"{BASE_FOLDER}/Augmented Images/Pre-_Evening_Fire_Incident_aug_img/Pre-_Evening_Fire_Incident_aug_img": 1,
    f"{BASE_FOLDER}/Augmented Images/Evening_Forest_Condition_aug_img/Evening_Forest_Condition_aug_img": 0,
    f"{BASE_FOLDER}/Augmented Images/Pre-_Evening_Forest_Condition_aug_img/Pre-_Evening_Forest_Condition_aug_img": 0,
}

all_images, all_labels, folder_counts = [], [], {}
for folder_path_suffix, label in FOLDER_LABEL_MAP.items():
    folder_path = os.path.join(CFG['dataset_root'], folder_path_suffix)
    if not os.path.exists(folder_path): continue
    imgs = (glob.glob(os.path.join(folder_path, '*.jpg')) +
            glob.glob(os.path.join(folder_path, '*.png')) +
            glob.glob(os.path.join(folder_path, '*.jpeg')))
    all_images.extend(imgs); all_labels.extend([label] * len(imgs))
    folder_counts[folder_path_suffix.split('/')[-1]] = len(imgs)

print(f"Total images: {len(all_images)}")
print(f"Fire (1):     {sum(all_labels)}")
print(f"Non-Fire (0): {len(all_labels) - sum(all_labels)}")

## Cell 5: Visualize Sample Images
**Lines 136–152** — Display 8 fire and 8 no-fire sample images in a grid

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fire_imgs   = [p for p, l in zip(all_images, all_labels) if l == 1]
nofire_imgs = [p for p, l in zip(all_images, all_labels) if l == 0]
random.shuffle(fire_imgs); random.shuffle(nofire_imgs)
for i in range(8):
    img = Image.open(fire_imgs[i]).convert('RGB').resize((112, 112))
    axes[0, i].imshow(img); axes[0, i].set_title('FIRE', color='red'); axes[0, i].axis('off')
    img = Image.open(nofire_imgs[i]).convert('RGB').resize((112, 112))
    axes[1, i].imshow(img); axes[1, i].set_title('NON-FIRE', color='green'); axes[1, i].axis('off')
plt.suptitle('Sample Images from UAVS-FDDB Dataset', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Cell 6: Stratified 70/15/15 Split
**Lines 159–162** — train_test_split twice to get train/val/test

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    all_images, all_labels,
    test_size=(CFG['val_ratio'] + CFG['test_ratio']),
    stratify=all_labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## Cell 7: DataLoaders
**Lines 169–204** — FireDataset class, augmentation transforms, WeightedRandomSampler

In [ ]:
class FireDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

train_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_test_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = FireDataset(X_train, y_train, train_transform)
val_dataset   = FireDataset(X_val,   y_val,   val_test_transform)
test_dataset  = FireDataset(X_test,  y_test,  val_test_transform)

class_counts = [y_train.count(0), y_train.count(1)]
weights  = [1.0 / class_counts[label] for label in y_train]
sampler  = WeightedRandomSampler(weights, num_samples=len(y_train), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CFG['batch_size'], shuffle=False,    num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CFG['batch_size'], shuffle=False,    num_workers=2, pin_memory=True)
print("DataLoaders ready.")

## Cell 8: MobileNetV3-Small Backbone
**Lines 211–224** — Pre-trained ImageNet backbone, outputs 576-d feature vector

In [ ]:
class MobileNetV3Backbone(nn.Module):
    def __init__(self, freeze_until=0):
        super().__init__()
        base = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.avgpool  = base.avgpool
        for i, layer in enumerate(list(self.features.children())):
            if i < freeze_until:
                for param in layer.parameters(): param.requires_grad = False

    def forward(self, x):
        return self.avgpool(self.features(x)).flatten(1)

backbone = MobileNetV3Backbone(freeze_until=CFG['freeze_until_layer']).to(DEVICE)
print("Backbone ready. Output: 576-d feature vector.")

## Cell 9: PennyLane Data Re-uploading Quantum Circuit
**Lines 231–253** — 8-qubit, 3-layer VQC with RY encoding + CNOT entanglement + trainable weights

In [ ]:
dev = qml.device("default.qubit", wires=CFG['n_qubits'])

@qml.qnode(dev, interface="torch")
def quantum_net(inputs, weights):
    n = CFG['n_qubits']
    for layer in range(CFG['n_reuploading_layers']):
        # Data Re-uploading: encode data into every layer
        for i in range(n): qml.RY(inputs[:, layer * n + i] * torch.pi, wires=i)
        # Entanglement: circular CNOT ring
        for i in range(n): qml.CNOT(wires=[i, (i + 1) % n])
        # Trainable rotation gates
        for i in range(n): qml.RY(weights[layer, i], wires=i)
    # Measure Pauli-Z expectation values for all qubits
    return [qml.expval(qml.PauliZ(i)) for i in range(n)]

class DataReuploadingQCircuit(nn.Module):
    def __init__(self, n_qubits=8, n_layers=3, feature_dim=576):
        super().__init__()
        # Project 576-d → 24 angles (8 qubits × 3 layers)
        self.pre_proj = nn.Sequential(
            nn.Linear(feature_dim, 128), nn.BatchNorm1d(128), nn.GELU(),
            nn.Linear(128, n_qubits * n_layers), nn.Tanh())
        self.var_weights = nn.Parameter(torch.randn(n_layers, n_qubits) * 0.01)
        # Project 8 expectation values → 16-d
        self.post_proj = nn.Sequential(nn.Linear(n_qubits, 16), nn.LayerNorm(16))

    def forward(self, x):
        angles = self.pre_proj(x)
        q_out  = quantum_net(angles, self.var_weights)
        quantum_features = torch.stack(q_out, dim=-1).to(x.device).float()
        return self.post_proj(quantum_features)

print("Quantum circuit defined.")

## Cell 10: Complete Hybrid Model
**Lines 260–272** — Combines backbone + quantum circuit + MLP classifier

In [ ]:
class HybridQuantumFireDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone   = MobileNetV3Backbone(freeze_until=CFG['freeze_until_layer'])
        self.quantum    = DataReuploadingQCircuit(
            n_qubits=CFG['n_qubits'],
            n_layers=CFG['n_reuploading_layers'],
            feature_dim=CFG['backbone_feature_dim'])
        self.classifier = nn.Sequential(
            nn.Linear(16, 64), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(64, CFG['num_classes']))
        self.gradcam_target_layer = self.backbone.features[-1]

    def forward(self, x):
        return self.classifier(self.quantum(self.backbone(x)))

model = HybridQuantumFireDetector().to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f"Model Built. Total Params: {total:,}")

## Cell 11: Loss, Optimizer, Scheduler Setup
**Lines 279–283** — CrossEntropyLoss + AdamW + CosineAnnealingLR

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)

history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
           'val_loss':   [], 'val_acc':   [], 'val_f1':   []}
print("Optimizer and scheduler ready.")

## Cell 12: Training Functions
**Lines 290–319** — `train_one_epoch()` and `validate()` functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0; all_preds, all_labels = [], []
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0; all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds); all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds)

print("Training functions defined.")

## Cell 13: ⚡ TRAINING LOOP (Main Training Cell)
**Lines 326–348** — This is the actual training execution.

### Exact Line-by-Line Breakdown:
| Line | Code | Purpose |
|------|------|---------|
| 327 | `best_val_f1 = 0.0` | Track best F1 for checkpoint saving |
| 329 | `for epoch in range(CFG['epochs']):` | Loop over 15 epochs |
| 331 | `train_one_epoch(...)` | Run one full pass over training data |
| 332 | `validate(...)` | Evaluate on validation set |
| 333 | `scheduler.step()` | Decay learning rate via Cosine Annealing |
| 340-342 | `if val_f1 > best_val_f1` | Save checkpoint if best model so far |
| 345-346 | `patience_counter >= 5` | Early stopping if no improvement for 5 epochs |

In [ ]:
# ═══════════════════════════════════════════════════
#  TRAINING LOOP — Cell 13 | Lines 326-348
# ═══════════════════════════════════════════════════
print(f"Starting training on {DEVICE}...")
best_val_f1 = 0.0
patience_counter = 0
start_time = time.time()

for epoch in range(CFG['epochs']):                                          # Line 329: epoch loop
    print(f"\nEpoch {epoch+1}/{CFG['epochs']}")
    train_loss, train_acc, train_f1 = train_one_epoch(                     # Line 331: train
        model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, val_f1 = validate(                                   # Line 332: validate
        model, val_loader, criterion, DEVICE)
    scheduler.step()                                                         # Line 333: LR decay

    for k, v in zip(history.keys(),
                    [train_loss, train_acc, train_f1, val_loss, val_acc, val_f1]):
        history[k].append(v)

    print(f"  Train -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   -> Loss: {val_loss:.4f}   | Acc: {val_acc:.4f}   | F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:                                                # Line 340: save best
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), CFG['checkpoint_path'])              # Line 341: checkpoint
        print(f"  --> Saved new best model! (Val F1: {best_val_f1:.4f})")
    else:
        patience_counter += 1

    if patience_counter >= 5:                                                # Line 345: early stop
        print(f"\nEarly stopping after {epoch+1} epochs!"); break

print(f"\nTraining Complete in {(time.time()-start_time)/60:.1f} minutes!")

## Cell 14: Final Test Set Evaluation
**Lines 355–371** — Load best checkpoint, evaluate on unseen test set, print all metrics

In [ ]:
print("Loading best model for testing...")
model.load_state_dict(torch.load(CFG['checkpoint_path']))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs); all_preds.extend(preds); all_labels.extend(labels.cpu().numpy())

print(f"\n{'='*40}")
print(f"  FINAL TEST SET METRICS")
print(f"{'='*40}")
print(f"  Accuracy  : {accuracy_score(all_labels, all_preds)*100:.2f}%")
print(f"  Precision : {precision_score(all_labels, all_preds)*100:.2f}%")
print(f"  Recall    : {recall_score(all_labels, all_preds)*100:.2f}%")
print(f"  F1 Score  : {f1_score(all_labels, all_preds)*100:.2f}%")
print(f"  AUC-ROC   : {roc_auc_score(all_labels, all_probs):.4f}")
print(f"{'='*40}")

## Cell 15: Export All Artifacts (Graphs, Heatmaps, ZIP)
**Lines 378–434** — Saves training curves, confusion matrix, ROC/PR curves, Grad-CAM heatmaps, and ZIP

In [ ]:
base_dir = "Capstone_Outputs"
dirs = ["Models", "Dataset_Analysis", "Training_Results", "Visualizations", "Deep_Analysis"]
for d in dirs: os.makedirs(os.path.join(base_dir, d), exist_ok=True)

# 1. Training Graphs
epochs_range = range(1, len(history['train_loss']) + 1)
for metric, fname in [('loss', 'Loss_Curve'), ('acc', 'Accuracy_Curve'), ('f1', 'F1_Curve')]:
    plt.figure(figsize=(8, 6))
    plt.plot(epochs_range, history[f'train_{metric}'], label='Train')
    plt.plot(epochs_range, history[f'val_{metric}'],   label='Val')
    plt.title(fname.replace('_', ' ')); plt.legend()
    plt.savefig(f"{base_dir}/Training_Results/{fname}.png", dpi=200); plt.close()

# 2. Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Fire','Fire'], yticklabels=['Non-Fire','Fire'])
plt.title('Test Set Confusion Matrix')
plt.savefig(f"{base_dir}/Training_Results/Confusion_Matrix.png", dpi=200); plt.close()

# 3. ROC & PR Curves
fpr, tpr, _ = roc_curve(all_labels, all_probs)
prec_vals, rec_vals, _ = precision_recall_curve(all_labels, all_probs)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC={auc(fpr,tpr):.4f})')
axes[0].plot([0,1],[0,1], 'navy', lw=2, linestyle='--'); axes[0].set_title('ROC Curve'); axes[0].legend()
axes[1].plot(rec_vals, prec_vals, 'green', lw=2, label=f'PR (AUC={auc(rec_vals,prec_vals):.4f})')
axes[1].set_title('PR Curve'); axes[1].legend()
plt.savefig(f"{base_dir}/Deep_Analysis/ROC_PR_Curves.png", dpi=200); plt.close()

# 4. Grad-CAM Scenarios
model.eval()
cam_extractor = GradCAM(model, target_layer=model.gradcam_target_layer)

def generate_heatmap_fig(img_path, title_prefix):
    img_pil    = Image.open(img_path).convert('RGB').resize((CFG['img_size'], CFG['img_size']))
    img_tensor = val_test_transform(img_pil).unsqueeze(0).to(DEVICE)
    out   = model(img_tensor)
    prob  = torch.softmax(out, dim=1)[0, 1].item()
    cam   = cam_extractor(out.argmax(dim=1).item(), out)[0].squeeze(0).cpu().numpy()
    heatmap = cv2.resize(cv2.cvtColor(cv2.applyColorMap(
        np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB), (CFG['img_size'], CFG['img_size']))
    img_np  = np.array(img_pil)
    overlay = (0.6 * img_np + 0.4 * heatmap).astype(np.uint8)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_np);    axes[0].axis('off'); axes[0].set_title('Original')
    axes[1].imshow(cam,cmap='jet'); axes[1].axis('off'); axes[1].set_title('Grad-CAM')
    axes[2].imshow(overlay);  axes[2].axis('off'); axes[2].set_title(f'Overlay Conf:{prob*100:.1f}%')
    plt.suptitle(title_prefix, fontsize=14, fontweight='bold')
    return fig

scenarios = {
    'Pre-Evening_Fire':   'Pre-Evening Fire Incident_raw_img',
    'Evening_Fire':       'Evening Fire Incident_raw_img',
    'Pre-Evening_Forest': 'Pre-evening Forest condition_raw_img',
    'Evening_Forest':     'Evening Forest condition_raw_img',
}
for name, search_str in scenarios.items():
    matches = [p for p in X_test if search_str in p]
    if matches:
        fig = generate_heatmap_fig(matches[0], name.replace('_',' '))
        fig.savefig(f"{base_dir}/Deep_Analysis/Scenario_{name}.png", dpi=200); plt.close(fig)

cam_extractor.remove_hooks()

# 5. Save model & ZIP
if os.path.exists('best_model.pt'):
    shutil.move('best_model.pt', f"{base_dir}/Models/best_model.pt")
shutil.make_archive('Final_Capstone_Artifacts', 'zip', base_dir)

print("\n✅ ALL ARTIFACTS GENERATED AND ZIPPED SUCCESSFULLY!")
print("Download 'Final_Capstone_Artifacts.zip' from the Output panel!")